# drive2sim run $run_id

Notebook do `d2s kaggle push` sinh ra. Chạy từ trên xuống; kết quả ghi vào `/kaggle/working/runs/` và `/kaggle/working/cache/`, tải về bằng `d2s kaggle pull $run_id`.

In [ ]:
# Môi trường của phiên Kaggle: ghi lại để điền câu hỏi mở về phiên bản PyTorch.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch

print("torch", torch.__version__, "cuda", torch.version.cuda)

In [ ]:
# Lấy đúng commit đã tạo run.
!git clone --quiet --recurse-submodules --branch $branch $repo_url /kaggle/tmp/repo
%cd /kaggle/tmp/repo
!git checkout --quiet $commit && git submodule update --quiet --init --recursive

In [ ]:
# Cài phụ thuộc. Giữ PyTorch có sẵn của Kaggle: bỏ các dòng torch, torchvision, xformers và
# --extra-index-url trong requirements.txt. Chưa chạy thử; sửa ở drive2sim/kaggle/templates/run.ipynb.
!grep -vE '^(torch|torchvision|xformers)==|^--extra-index-url' requirements.txt > /kaggle/tmp/requirements.txt
!pip install --quiet -r /kaggle/tmp/requirements.txt
!pip install --quiet -e .[studio]

In [ ]:
# Gói job (CONTRACTS mục 9.2).
import json
import pathlib

bundle = json.loads(r"""$bundle_json""")
bundle_dir = pathlib.Path("/kaggle/tmp/bundle")
bundle_dir.mkdir(parents=True, exist_ok=True)
for name, content in bundle.items():
    (bundle_dir / name).write_text(json.dumps(content, indent=2))

In [ ]:
!d2s kaggle run /kaggle/tmp/bundle --workdir /kaggle/working --input-root /kaggle/input

In [ ]:
# Output chỉ giữ runs/ và cache/: bỏ data/ (symlink tới dataset) và cảnh ảo bằng symlink.
!rm -rf /kaggle/working/data
!find /kaggle/working/runs /kaggle/working/cache -type l -delete 2>/dev/null; true